In [3]:
from langchain.chat_models import ChatOpenAI, ChatOllama, ChatAnthropic
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler


chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)


# t = PromptTemplate(
#     input_variables=["country"],
#     template="What is the capital of {country}?",
# )

t = PromptTemplate.from_template("What is the capital of {country}?")

t.format(country="France")

/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_65440/1584664218.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat = ChatOpenAI(


'What is the capital of France?'

In [4]:
examples = [
    {
        "country": "France",
        "answer": """
Here is what I know:
Capital: Paris
Language: French
Food: Wine and Cheese
Currency: Euro
""",
    },
    {
        "country": "Italy",
        "answer": """
I know this:
Capital: Rome
Language: Italian
Food: Pizza and Pasta
Currency: Euro
""",
    },
    {
        "country": "Greece",
        "answer": """
I know this:
Capital: Athens
Language: Greek
Food: Souvlaki and Feta Cheese
Currency: Euro
""",
    },
]

In [5]:
chat.predict("what do you know about France?")

/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_65440/3452864850.py:1: LangChainDeprecationWarning: The method `BaseChatModel.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chat.predict("what do you know about France?")


France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. France is also known for its wine production, fashion industry, and art scene.

The country has a population of around 67 million people and is a member of the European Union. French is the official language, and the currency is the Euro. France has a diverse landscape, including mountains, beaches, and countryside.

France has a long history of influential art movements, such as Impressionism and Surrealism, and has produced many famous artists, writers, and thinkers. The country is also known for its strong emphasis on food and wine, with French cuisine being highly regarded worldwide.

France has a strong economy, with industries such as aerospace, automotive, and luxury goods playing a significant role. The country is also a popular tourist destination, attra

'France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. France is also known for its wine production, fashion industry, and art scene.\n\nThe country has a population of around 67 million people and is a member of the European Union. French is the official language, and the currency is the Euro. France has a diverse landscape, including mountains, beaches, and countryside.\n\nFrance has a long history of influential art movements, such as Impressionism and Surrealism, and has produced many famous artists, writers, and thinkers. The country is also known for its strong emphasis on food and wine, with French cuisine being highly regarded worldwide.\n\nFrance has a strong economy, with industries such as aerospace, automotive, and luxury goods playing a significant role. The country is also a popular tourist destination

In [6]:
example_template = """ 
    Human: {country}
    AI: {answer}
"""
example_promt = PromptTemplate.from_template(example_template)
# example_promt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

prompt = FewShotPromptTemplate(
    example_prompt=example_promt,
    examples=examples,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
    # prefix="You are a helpful assistant.",
)

# prompt.format(country="Germany")

chain = prompt | chat

# delete metadata, id

chain.invoke({"country": "Germany"})

AI: 
I know this:
Capital: Berlin
Language: German
Food: Bratwurst and Sauerkraut
Currency: Euro

AIMessage(content='AI: \nI know this:\nCapital: Berlin\nLanguage: German\nFood: Bratwurst and Sauerkraut\nCurrency: Euro', additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-9384f911-241f-4ce7-b4ab-2074cd0586b1-0')

In [7]:
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.prompts import ChatPromptTemplate

In [8]:
# more intuitive way to create chat prompts

example_promt = ChatPromptTemplate.from_messages(
    [
        ("human", "{country}"),
        ("ai", "{answer}"),
    ]
)

example_prompt = FewShotChatMessagePromptTemplate(
    examples=examples, example_prompt=example_promt
)
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a geography expert, after the template answers, you give extra long answers",
        ),
        example_prompt,
        ("human", "What do you know about {country}?"),
    ]
)

chain = final_prompt | chat

chain.invoke({"country": "Germany"})

I know this:
Capital: Berlin
Language: German
Food: Bratwurst and Sauerkraut
Currency: Euro

Germany is a country located in central Europe known for its rich history, culture, and strong economy. It is the most populous country in the European Union and is known for its technological advancements, high-quality engineering, and efficient transportation systems. Germany is also famous for its beer culture, with Oktoberfest being a popular annual festival in Munich. The country is home to many historic landmarks, such as the Brandenburg Gate in Berlin and the Neuschwanstein Castle in Bavaria. Germany is also known for its beautiful landscapes, including the Black Forest, the Rhine River Valley, and the Bavarian Alps.

AIMessage(content='I know this:\nCapital: Berlin\nLanguage: German\nFood: Bratwurst and Sauerkraut\nCurrency: Euro\n\nGermany is a country located in central Europe known for its rich history, culture, and strong economy. It is the most populous country in the European Union and is known for its technological advancements, high-quality engineering, and efficient transportation systems. Germany is also famous for its beer culture, with Oktoberfest being a popular annual festival in Munich. The country is home to many historic landmarks, such as the Brandenburg Gate in Berlin and the Neuschwanstein Castle in Bavaria. Germany is also known for its beautiful landscapes, including the Black Forest, the Rhine River Valley, and the Bavarian Alps.', additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-6a9a5721-2019-4320-b882-4fcd64627801-0')

In [9]:
from langchain.prompts.example_selector import LengthBasedExampleSelector

example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=100,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")

'Human: France\nAI: \nHere is what I know:\nCapital: Paris\nLanguage: French\nFood: Wine and Cheese\nCurrency: Euro\n\n\nHuman: Italy\nAI: \nI know this:\nCapital: Rome\nLanguage: Italian\nFood: Pizza and Pasta\nCurrency: Euro\n\n\nHuman: Greece\nAI: \nI know this:\nCapital: Athens\nLanguage: Greek\nFood: Souvlaki and Feta Cheese\nCurrency: Euro\n\n\nHuman: What do you know about Brazil?'

In [10]:
from langchain.prompts.example_selector.base import BaseExampleSelector


class RandomExampleSelector(BaseExampleSelector):
    def __init__(self, examples):
        self.examples = examples

    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice

        return [choice(self.examples)]

In [11]:
example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")

'Human: France\nAI: \nHere is what I know:\nCapital: Paris\nLanguage: French\nFood: Wine and Cheese\nCurrency: Euro\n\n\nHuman: What do you know about Brazil?'

In [12]:
from langchain.prompts import load_prompt

prompt = load_prompt("./prompt.json")


prompt.format(country="Germany")

'What is the capital of Germany?'

In [13]:
prompt = load_prompt("./prompt.yaml")
prompt.format(country="Germany")

'What is the capital of Germany?'

In [ ]:
from langchain.prompts.pipeline import PipelinePromptTemplate

intro = PromptTemplate.from_template(
    """
You are a role playing assistant.
And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
This is an example of how you talk:

Human: {example_question}
You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
Start now!

Human: {question}
You:
"""
)

final = PromptTemplate.from_template(
    """
{intro}

{example}

{start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start),
]

In [15]:
full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)


chain = full_prompt | chat

chain.invoke(
    {
        "character": "Pirate",
        "example_question": "What is your location?",
        "example_answer": "Arrrrg! That is a secret!! Arg arg!!",
        "question": "What is your fav food?",
    }
)

/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_65440/2222041834.py:1: LangChainDeprecationWarning: This class is deprecated. Please see the docstring below or at the link for a replacement option: https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.pipeline.PipelinePromptTemplate.html
  full_prompt = PipelinePromptTemplate(


Arrr matey! Me favorite grub be a hearty plate o' salted beef and hardtack! Aye, it be the sustenance of true pirates on the high seas! Arrr!

AIMessage(content="Arrr matey! Me favorite grub be a hearty plate o' salted beef and hardtack! Aye, it be the sustenance of true pirates on the high seas! Arrr!", additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-d695fc72-6fa3-4f96-b23a-db268adcb7d6-0')

In [19]:
# for saving money on API calls
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache, SQLiteCache


# set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache(database_path="cache.db"))


chat = ChatOpenAI(
    temperature=0.1,
    # streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
)



# chat.predict("How do you make italian pasta?")

In [20]:
chat.predict("How do you make italian pasta?")
# takes no time, because it is cached

'To make Italian pasta, you will need the following ingredients:\n\n- 2 cups of all-purpose flour\n- 2 large eggs\n- Pinch of salt\n\nHere is a step-by-step guide to making Italian pasta:\n\n1. On a clean work surface, pour the flour and create a well in the center.\n2. Crack the eggs into the well and add a pinch of salt.\n3. Using a fork, gradually mix the eggs into the flour until a dough forms.\n4. Knead the dough for about 10 minutes until it is smooth and elastic.\n5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.\n6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.\n7. Cut the dough into desired shapes, such as fettuccine or spaghetti.\n8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes, or until al dente.\n9. Drain the pasta and toss with your favorite sauce or toppings.\n\nEnjoy your homemade Italian pasta!'

In [21]:
from langchain.callbacks import get_openai_callback

with get_openai_callback() as usage:
    chat.predict("What is the recipe for soju?")
    print(usage)

Tokens Used: 214
	Prompt Tokens: 15
		Prompt Tokens Cached: 0
	Completion Tokens: 199
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.00042050000000000003


In [ ]:
with get_openai_callback() as usage:
    a = chat.predict ("What is the recipe for soju")
    b = chat.predict("What is the recipe for bread")
    print(a,b, "\n")
    print(usage)
    # print(usage.total_cost, usage.total_tokens, usage.prompt_tokens, usage.completion_tokens)

Ingredients:
- 1 cup of rice
- 1 cup of water
- 1 tablespoon of nuruk (fermentation starter)
- 1 tablespoon of sugar

Instructions:
1. Rinse the rice thoroughly and soak it in water for at least 1 hour.
2. Drain the rice and steam it until fully cooked.
3. Let the rice cool down to room temperature.
4. In a large bowl, mix the nuruk and sugar with the cooked rice.
5. Cover the bowl with a clean cloth and let it ferment in a warm place for 3-4 days.
6. After fermentation, strain the mixture through a cheesecloth to remove any solids.
7. Transfer the liquid to a clean bottle and store it in the refrigerator.
8. Serve the homemade soju chilled and enjoy responsibly. Ingredients:
- 4 cups all-purpose flour
- 1 packet active dry yeast
- 1 1/2 cups warm water
- 2 tablespoons sugar
- 2 teaspoons salt
- 2 tablespoons olive oil

Instructions:
1. In a large mixing bowl, combine the warm water, sugar, and yeast. Let it sit for about 5-10 minutes until the yeast is foamy.
2. Add the flour, salt, a

In [23]:
#serializing the model
from langchain.llms.openai import OpenAI

chat = OpenAI(temperature=0.1, max_tokens=450, model="gpt-3.5-turbo-16k")

chat.save("model.json" )

/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_65440/3183447838.py:4: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  chat = OpenAI(temperature=0.1, max_tokens=450, model="gpt-3.5-turbo-16k")


In [26]:
from langchain.llms.loading import load_llm
chat = load_llm("model.json")
chat

/Users/tae/anaconda3/envs/gpt-full/lib/python3.11/site-packages/langchain_community/llms/openai.py:255: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
/Users/tae/anaconda3/envs/gpt-full/lib/python3.11/site-packages/langchain_community/llms/openai.py:1089: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(


OpenAIChat(client=<class 'openai.api_resources.chat_completion.ChatCompletion'>, model_kwargs={'model_name': 'gpt-3.5-turbo-16k', 'temperature': 0.1, 'top_p': 1.0, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'n': 1, 'logit_bias': {}, 'max_tokens': 450}, prefix_messages=[])